<a href="https://colab.research.google.com/github/marsya505/DataMining/blob/main/Week13-ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor

# Import necessary modules
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt
from sklearn.metrics import r2_score

In [ ]:
df = pd.read_csv('diabetes.csv')
print(df.shape)
df.describe().transpose()

In [ ]:
target_column = ['Outcome']
predictors = list(set(list(df.columns))-set(target_column))
df[predictors] = df[predictors]/df[predictors].max()
df.describe().transpose()

In [ ]:
X = df[predictors].values
y = df[target_column].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=40)
print(X_train.shape); print(X_test.shape)

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(8,8,8), activation='relu', solver='adam', max_iter=500)
mlp.fit(X_train, y_train)

predict_train = mlp.predict(X_train)
predict_test = mlp.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_train, predict_train))
print(classification_report(y_train, predict_train))

In [ ]:
hidden_layers = [
    (10, 10, 6),
    (6, 8, 9),
    (6, 9, 1),
    (6, 3, 9),
    (7, 10, 8),
]

results_hl = []

for hl in hidden_layers:
    mlp = MLPClassifier(
        hidden_layer_sizes=hl,
        activation='relu',
        solver='adam',
        max_iter=500,
        random_state=42
    )
    mlp.fit(X_train, y_train)

    pred_train = mlp.predict(X_train)
    pred_test  = mlp.predict(X_test)

    acc_train = accuracy_score(y_train, pred_train)
    acc_test  = accuracy_score(y_test, pred_test)

    results_hl.append({
        'Hidden Layer': str(hl),
        'Acc Train': acc_train,
        'Acc Test':  acc_test
    })

    label = "baseline" if hl == (8, 8, 8) else ""
    print(f"\nHidden Layer Size: {hl}  {label}")
    print(f"  Training Accuracy : {acc_train:.4f} ({acc_train*100:.2f}%)")
    print(f"  Testing  Accuracy : {acc_test:.4f} ({acc_test*100:.2f}%)")
    print("  Confusion Matrix (Test):")
    print(f"  {confusion_matrix(y_test, pred_test)}")
    print("  Classification Report (Test):")
    print(classification_report(y_test, pred_test, zero_division=0))

# Ringkasan hidden layer
df_hl = pd.DataFrame(results_hl).sort_values('Acc Test', ascending=False)
print(df_hl.to_string(index=False))
best_hl = df_hl.iloc[0]
print(f"\nTERBAIK: {best_hl['Hidden Layer']} | Test Acc: {best_hl['Acc Test']*100:.2f}%")

In [ ]:
activations = ['relu', 'tanh', 'logistic', 'identity']
results_act = []

for act in activations:
    mlp = MLPClassifier(
        hidden_layer_sizes=(8, 8, 8),
        activation=act,
        solver='adam',
        max_iter=500,
        random_state=42
    )
    mlp.fit(X_train, y_train)

    pred_train = mlp.predict(X_train)
    pred_test  = mlp.predict(X_test)

    acc_train = accuracy_score(y_train, pred_train)
    acc_test  = accuracy_score(y_test, pred_test)

    results_act.append({
        'Activation': act,
        'Acc Train': acc_train,
        'Acc Test':  acc_test
    })

    print(f"\nActivation: {act}")
    print(f"  Training Accuracy : {acc_train:.4f} ({acc_train*100:.2f}%)")
    print(f"  Testing  Accuracy : {acc_test:.4f} ({acc_test*100:.2f}%)")
    print("  Classification Report (Test):")
    print(classification_report(y_test, pred_test, zero_division=0))

# Ringkasan activation
df_act = pd.DataFrame(results_act).sort_values('Acc Test', ascending=False)
print("RINGKASAN — ACTIVATION FUNCTION (diurutkan test accuracy)")
print(df_act.to_string(index=False))
best_act = df_act.iloc[0]
print(f"\nTERBAIK: {best_act['Activation']} | Test Acc: {best_act['Acc Test']*100:.2f}%")

# VISUALISASI
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Eksperimen ANN - Dataset Diabetes', fontsize=14, fontweight='bold')

# Plot 1: Hidden Layer Size
ax1 = axes[0]
labels_hl = [r['Hidden Layer'] for r in results_hl]
train_hl  = [r['Acc Train'] * 100 for r in results_hl]
test_hl   = [r['Acc Test']  * 100 for r in results_hl]

x = np.arange(len(labels_hl))
w = 0.35
bars1 = ax1.bar(x - w/2, train_hl, w, label='Train', color='steelblue', alpha=0.85)
bars2 = ax1.bar(x + w/2, test_hl,  w, label='Test',  color='coral',     alpha=0.85)

# Tandai bar terbaik
best_idx = test_hl.index(max(test_hl))
bars2[best_idx].set_edgecolor('green')
bars2[best_idx].set_linewidth(2.5)

ax1.set_title('Perbandingan Hidden Layer Size')
ax1.set_xlabel('Hidden Layer Size')
ax1.set_ylabel('Accuracy (%)')
ax1.set_xticks(x)
ax1.set_xticklabels(labels_hl, rotation=20, ha='right', fontsize=9)
ax1.set_ylim(40, 80)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=7)

# Plot 2: Activation Function
ax2 = axes[1]
labels_act = [r['Activation'] for r in results_act]
train_act  = [r['Acc Train'] * 100 for r in results_act]
test_act   = [r['Acc Test']  * 100 for r in results_act]

x2 = np.arange(len(labels_act))
bars3 = ax2.bar(x2 - w/2, train_act, w, label='Train', color='steelblue', alpha=0.85)
bars4 = ax2.bar(x2 + w/2, test_act,  w, label='Test',  color='coral',     alpha=0.85)

best_idx2 = test_act.index(max(test_act))
bars4[best_idx2].set_edgecolor('green')
bars4[best_idx2].set_linewidth(2.5)

ax2.set_title('Perbandingan Activation Function')
ax2.set_xlabel('Activation Function')
ax2.set_ylabel('Accuracy (%)')
ax2.set_xticks(x2)
ax2.set_xticklabels(labels_act, fontsize=10)
ax2.set_ylim(40, 80)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
for bar in bars3:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
for bar in bars4:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('hasil_eksperimen_ann.png', dpi=150, bbox_inches='tight')
plt.show()